In [1]:
# =============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# =============================================================================
# We group all imports upfront so any missing dependency is caught immediately
# before any time-consuming computation begins.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Scikit-Learn: classical machine learning tools
from sklearn.model_selection import TimeSeriesSplit          # Time-aware CV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer               # Apply different transforms per column type
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# PyTorch: deep learning backbone
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# --- Global settings ---
# 'ggplot' is a clean, readable style similar to R's ggplot2
plt.style.use('ggplot')

# Create output directory for plots (exist_ok=True avoids crash if already present)
os.makedirs('plots', exist_ok=True)

# Fix random seeds for full reproducibility: same results every run
np.random.seed(42)
torch.manual_seed(42)

# The number of consecutive hours fed as input to the CNN
# 24 = one full day of historical context to predict the next hour
SEQUENCE_LENGTH = 24




ModuleNotFoundError: No module named 'torch'

In [ ]:
# =============================================================================
# CELL 2: DATA LOADING AND CLEANING
# =============================================================================
# Raw CSV → clean DataFrame.
# We immediately fix known data quality issues before any analysis.

print("--> CELL 2: Loading and Cleaning Data...")

df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")

# Parse the date column from its non-standard DD-MM-YYYY HH:MM string format
# into proper datetime objects so we can extract hour, month, etc. later
df['date_time'] = pd.to_datetime(df['date_time'], format='%d-%m-%Y %H:%M')

# A temperature of 0 K (absolute zero, −273 °C) is physically impossible and
# is a known sensor error in this dataset — we drop those rows entirely
df = df[df['temp'] > 0].copy()




In [ ]:
# =============================================================================
# CELL 3: EXPLORATORY DATA ANALYSIS (EDA) AND VISUALIZATION
# =============================================================================
# Goal: understand traffic patterns BEFORE building any model.
# Three plots: (1) hourly distribution, (2) weather impact at rush hour,
# (3) temperature vs volume coloured by time-of-day.

print("--> CELL 3: Generating EDA Plots...")

# Extract the hour (0-23) for grouping — needed for all three plots
df['hour'] = df['date_time'].dt.hour

# Bin hours into four named periods to produce readable legend entries
df['time_of_day'] = pd.cut(
    df['hour'],
    bins=[-1, 6, 12, 18, 24],
    labels=['Night (0-6)', 'Morning (6-12)', 'Afternoon (12-18)', 'Evening (18-24)']
)

# --- Plot 1: Traffic distribution per hour ---
# A box plot shows median + spread, revealing the classic twin rush-hour peaks
plt.figure(figsize=(12, 6))
sns.boxplot(x='hour', y='traffic_volume', data=df, palette='viridis')
plt.title("Baseline: Traffic Volume Distribution by Hour of the Day")
plt.xlabel("Hour of Day (0-23)")
plt.ylabel("Traffic Volume")
plt.savefig('plots/eda_traffic_by_hour.png')
plt.close()

# --- Plot 2: Weather impact at rush hours only ---
# Rush hours (7-9 AM, 4-6 PM) are when weather influence is most visible;
# analysing all hours together would bury the signal in off-peak noise
rush_hour_mask = df['hour'].isin([7, 8, 9, 16, 17, 18])
df_rush = df[rush_hour_mask]

plt.figure(figsize=(14, 7))
sns.boxplot(x='weather_main', y='traffic_volume', data=df_rush, palette='Set2')
plt.title("Traffic Volume vs Weather Conditions (Rush Hours Only: 7-9h & 16-18h)")
plt.xlabel("Main Weather Category")
plt.ylabel("Traffic Volume")
plt.xticks(rotation=45)
plt.savefig('plots/eda_traffic_vs_weather_rushhour.png')
plt.close()

# --- Plot 3: Temperature vs volume, coloured by time-of-day ---
# We sample 10 000 points to avoid rendering millions of overlapping dots
df_sample = df.sample(n=min(10000, len(df)), random_state=42)

plt.figure(figsize=(10, 8))
sns.scatterplot(x='temp', y='traffic_volume', hue='time_of_day',
                data=df_sample, alpha=0.5, palette='coolwarm')
plt.title("Traffic Volume vs Temperature (Kelvin)")
plt.xlabel("Temperature (K)")
plt.ylabel("Traffic Volume")
plt.legend(title='Time of Day')
plt.savefig('plots/eda_traffic_vs_temperature.png')
plt.close()




In [ ]:
# =============================================================================
# CELL 4: FEATURE ENGINEERING
# =============================================================================
# We derive new columns that encode domain knowledge the raw data doesn't
# expose directly (e.g., cyclical time encoding, holiday flag).

print("--> CELL 4: Feature Engineering...")

# Day-of-week (0=Monday … 6=Sunday) and month capture weekly/seasonal rhythms
df['day_of_week'] = df['date_time'].dt.dayofweek
df['month']       = df['date_time'].dt.month

# Binary flag: any named US holiday is encoded as 1, otherwise 0
# The holiday column contains the holiday name as a string, or the word 'None'
df['is_holiday'] = (df['holiday'] != 'None').astype(int)

# --- Cyclical encoding for hour ---
# A plain integer (0 to 23) implies 23 and 0 are far apart, but midnight is
# actually continuous. Projecting onto a unit circle fixes this:
#   sin(hour) + cos(hour) together uniquely represent every hour AND preserve
#   the wrap-around continuity (23h → 0h gap disappears).
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)




In [ ]:
# =============================================================================
# CELL 5: CHRONOLOGICAL SPLITTING AND PREPROCESSING
# =============================================================================
# WHY chronological split instead of random?
# Traffic data is ordered in time. A random split would allow the model to
# "see the future" during training (data leakage), yielding over-optimistic
# scores. We split at a fixed date to simulate real deployment conditions.

print("--> CELL 5: Chronological Splitting and Preprocessing...")

# Sort by time — mandatory before any time-based operation
df = df.sort_values('date_time').reset_index(drop=True)

# Everything before this date = training/validation; from this date = hold-out
split_date  = pd.to_datetime('2018-09-25')
future_mask = df['date_time'] >= split_date

df_train  = df[~future_mask].copy()   # Past data — used to train & cross-validate
df_future = df[ future_mask].copy()   # Future data — untouched until final evaluation

print(f"    [+] Training/Validation Data (before {split_date.date()}): {len(df_train)} rows.")
print(f"    [+] Hold-out Future Data    (after  {split_date.date()}): {len(df_future)} rows.")

# --- Feature lists ---
# OneHotEncoder handles the string weather category; StandardScaler handles numbers
categorical_features = ['weather_main']
numerical_features   = [
    'temp', 'rain_1h', 'snow_1h', 'clouds_all',
    'day_of_week', 'month', 'hour_sin', 'hour_cos', 'is_holiday'
]
all_features = numerical_features + categorical_features

# Extract raw feature matrices and target vectors
X_train_raw  = df_train[all_features]
y_train_raw  = df_train['traffic_volume'].values
X_future_raw = df_future[all_features]
y_future_raw = df_future['traffic_volume'].values

# --- Preprocessing pipeline ---
# ColumnTransformer applies different transforms to different column types in
# one step, then concatenates the results into a single NumPy array.
# IMPORTANT: fit_transform on training data only, then transform (not fit) on
# future data — the scaler parameters must come from the past, not the future.
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                                          numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
])

X_train_proc  = preprocessor.fit_transform(X_train_raw)   # Fit + transform (training)
X_future_proc = preprocessor.transform(X_future_raw)      # Transform only  (future)




In [ ]:
# =============================================================================
# CELL 6: MODEL DEFINITIONS
# =============================================================================
# We define three things here:
#   (A) A from-scratch gradient descent linear regressor (for pedagogical value)
#   (B) A 1-D CNN for time-series prediction (PyTorch)
#   (C) A helper that creates sliding-window sequences for the CNN
#   (D) The catalogue of classical ML models

print("--> CELL 6: Defining Models...")

# ---- (A) Gradient Descent Linear Regression (from scratch) ----
# This reimplements sklearn's LinearRegression using the explicit gradient
# update rule: θ ← θ − α · (1/m) · Xᵀ(Xθ − y)
# It exists to make the optimisation loop transparent and educational.
class GradientDescentLinearRegression:
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.learning_rate = learning_rate
        self.iterations    = iterations

    def fit(self, X, y):
        # Prepend a column of ones to X so the bias term θ₀ is learnable
        # without any special-casing — the dot product handles it uniformly
        X_b         = np.c_[np.ones((X.shape[0], 1)), X]
        self.theta  = np.zeros(X_b.shape[1])   # initialise all weights at 0
        m           = len(y)                    # number of training samples

        for _ in range(self.iterations):
            errors    = X_b.dot(self.theta) - y
            gradients = (1 / m) * X_b.T.dot(errors)   # mean squared gradient
            self.theta -= self.learning_rate * gradients

    def predict(self, X):
        X_b = np.c_[np.ones((X.shape[0], 1)), X]
        return X_b.dot(self.theta)


# ---- (B) 1-D Convolutional Neural Network ----
# Architecture: Conv1d → ReLU → MaxPool → Flatten → FC → ReLU → FC → scalar
#
# WHY Conv1d for time series?
#   A 1-D convolution slides a small kernel across the time axis and learns
#   local patterns (e.g., "traffic ramps up over 3 hours"). MaxPool halves the
#   temporal resolution, discarding fine noise while keeping salient structure.
#
# Input tensor shape: (batch, num_features, seq_length)
#   The feature dimension acts as the "channel" dimension — each feature has
#   its own lane, and the convolution mixes them across time.
class TS_CNN1D(nn.Module):
    def __init__(self, num_features, seq_length=24, out_channels=16):
        super(TS_CNN1D, self).__init__()

        # Kernel size 3 = look at 3 consecutive hours at once
        # padding=1 keeps the time dimension length equal to seq_length
        self.conv1 = nn.Conv1d(in_channels=num_features, out_channels=out_channels,
                               kernel_size=3, padding=1)
        self.relu  = nn.ReLU()

        # MaxPool(2) halves the sequence length → out_channels * (seq_length // 2) neurons after flatten
        self.pool  = nn.MaxPool1d(2)

        flattened_dim = out_channels * (seq_length // 2)
        self.fc1 = nn.Linear(flattened_dim, 32)
        self.fc2 = nn.Linear(32, 1)          # single output = predicted traffic volume

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # (batch, out_channels, seq_length//2)
        x = x.view(x.size(0), -1)                # flatten to (batch, flattened_dim)
        x = self.relu(self.fc1(x))
        return self.fc2(x)                        # (batch, 1)


# ---- (C) Sliding-window sequence builder ----
# The CNN needs fixed-length windows of consecutive hours.
# For each position i, X[i : i+seq_length] becomes one input sample and
# y[i+seq_length] is its label (the very next hour to predict).
def create_sequences(X, y, seq_length):
    xs, ys = [], []
    for i in range(len(X) - seq_length):
        xs.append(X[i : i + seq_length])
        ys.append(y[i + seq_length])
    return np.array(xs), np.array(ys)


# ---- (D) Reusable CNN training function ----
# Training the CNN requires the same boilerplate every time (DataLoader, Adam,
# loss loop). Extracting it into a function avoids copy-pasting the same 10
# lines in both the CV loop and the final evaluation.
def train_cnn(X_seq, y_seq, num_features, seq_length, epochs, batch_size=128):
    """
    Build and train a fresh TS_CNN1D from scratch.

    Parameters
    ----------
    X_seq        : np.ndarray, shape (N, seq_length, num_features)
    y_seq        : np.ndarray, shape (N,)
    num_features : int   — number of input features per timestep
    seq_length   : int   — length of each window
    epochs       : int   — number of full passes over the training data
    batch_size   : int   — mini-batch size for gradient updates

    Returns
    -------
    model : TS_CNN1D in eval mode, ready for inference
    """
    # Transpose to (N, num_features, seq_length) — Conv1d expects (batch, channels, length)
    X_t = torch.tensor(X_seq, dtype=torch.float32).transpose(1, 2)
    y_t = torch.tensor(y_seq, dtype=torch.float32).view(-1, 1)

    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

    model     = TS_CNN1D(num_features=num_features, seq_length=seq_length)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        for X_b, y_b in loader:
            optimizer.zero_grad()            # clear gradients from previous step
            loss = criterion(model(X_b), y_b)
            loss.backward()                  # compute gradients via backprop
            optimizer.step()                 # update weights with Adam

    model.eval()   # switch off dropout/batchnorm (not used here, but good practice)
    return model


# ---- (E) Classical ML model catalogue ----
# All four models share the same sklearn API: fit(X, y) / predict(X)
# This uniformity lets us loop over them without any special-casing.
models = {
    'Linear Regression': LinearRegression(),
    'Gradient Descent':  GradientDescentLinearRegression(learning_rate=0.1, iterations=500),
    'Random Forest':     RandomForestRegressor(n_estimators=50, max_depth=10,
                                               random_state=42, n_jobs=-1),
    'KNN':               KNeighborsRegressor(n_neighbors=10, weights='distance', n_jobs=-1),
}




In [ ]:
# =============================================================================
# CELL 7: TIME SERIES CROSS-VALIDATION (10 FOLDS)
# =============================================================================
# WHY TimeSeriesSplit instead of k-fold?
#   Standard k-fold shuffles data, so a test fold can contain samples from
#   before the training fold — data leakage again. TimeSeriesSplit always puts
#   the test fold AFTER the training fold, respecting temporal order.
#
# With 10 splits the training window grows progressively (expanding-window CV):
#   Fold 1: train on 10%, test on 10% | … | Fold 10: train on 90%, test on 10%
#
# This gives us 10 RMSE/R² estimates per model → we can report mean ± std.

print("--> CELL 7: TimeSeriesSplit Cross-Validation (10 Folds)...")

tscv = TimeSeriesSplit(n_splits=10)

# Accumulate per-fold metrics for every model, including the CNN
metrics_history = {name: {'RMSE': [], 'R2': []} for name in models}
metrics_history['CNN 1D (Sequence)'] = {'RMSE': [], 'R2': []}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X_train_proc), start=1):

    X_tr, X_te = X_train_proc[train_idx], X_train_proc[test_idx]
    y_tr, y_te = y_train_raw[train_idx],  y_train_raw[test_idx]

    # --- 7A: Evaluate classical models on this fold ---
    for name, model in models.items():
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)
        metrics_history[name]['RMSE'].append(np.sqrt(mean_squared_error(y_te, preds)))
        metrics_history[name]['R2'].append(r2_score(y_te, preds))

    # --- 7B: Evaluate CNN on this fold ---
    # Build windowed sequences from this fold's training and test arrays
    X_tr_seq, y_tr_seq = create_sequences(X_tr, y_tr, SEQUENCE_LENGTH)
    X_te_seq, y_te_seq = create_sequences(X_te, y_te, SEQUENCE_LENGTH)

    # The first few folds may be too small to produce any test sequences
    # (need at least SEQUENCE_LENGTH + 1 rows in the test set)
    if len(X_te_seq) > 0:
        # Train a fresh CNN for this fold (10 epochs is enough for quick CV)
        cnn = train_cnn(X_tr_seq, y_tr_seq,
                        num_features=X_train_proc.shape[1],
                        seq_length=SEQUENCE_LENGTH,
                        epochs=10)

        # Inference: no gradient computation needed → saves memory and time
        X_te_t = torch.tensor(X_te_seq, dtype=torch.float32).transpose(1, 2)
        with torch.no_grad():
            preds_cnn = cnn(X_te_t).numpy().flatten()

        metrics_history['CNN 1D (Sequence)']['RMSE'].append(
            np.sqrt(mean_squared_error(y_te_seq, preds_cnn))
        )
        metrics_history['CNN 1D (Sequence)']['R2'].append(
            r2_score(y_te_seq, preds_cnn)
        )
    else:
        # Record NaN so list lengths stay consistent across all models
        metrics_history['CNN 1D (Sequence)']['RMSE'].append(np.nan)
        metrics_history['CNN 1D (Sequence)']['R2'].append(np.nan)

    print(f"    [+] Fold {fold}/10 Completed.")




In [ ]:
# =============================================================================
# CELL 8: FINAL EVALUATION ON HOLD-OUT SET (PRODUCTION SIMULATION)
# =============================================================================
# Now we re-train every model on ALL available training data (no more CV folds)
# and evaluate once — and only once — on the hold-out future set.
#
# This final evaluation simulates real-world deployment:
#   - Model was built entirely on past data
#   - Performance is measured on genuinely unseen future data
#
# WHY re-train? The CV folds used only subsets; the final model benefits from
# seeing every available past sample before facing the future.

print("--> CELL 8: Final Evaluation on Hold-out Set (Future)...")

future_metrics = {}   # {model_name: {'R2': float, 'RMSE': float}}
future_preds   = {}   # {model_name: np.ndarray of predictions}

# --- 8A: Classical models — train on full training set, predict on future ---
for name, model in models.items():
    model.fit(X_train_proc, y_train_raw)
    preds = model.predict(X_future_proc)
    future_metrics[name] = {
        'R2':   r2_score(y_future_raw, preds),
        'RMSE': np.sqrt(mean_squared_error(y_future_raw, preds)),
    }
    future_preds[name] = preds

# --- 8B: CNN — train on full training sequences, predict on future sequences ---
X_full_seq, y_full_seq = create_sequences(X_train_proc,  y_train_raw,  SEQUENCE_LENGTH)
X_fut_seq,  y_fut_seq  = create_sequences(X_future_proc, y_future_raw, SEQUENCE_LENGTH)

# 15 epochs instead of 10 because we now have more data and more training time
final_cnn = train_cnn(X_full_seq, y_full_seq,
                      num_features=X_train_proc.shape[1],
                      seq_length=SEQUENCE_LENGTH,
                      epochs=15)

X_fut_t = torch.tensor(X_fut_seq, dtype=torch.float32).transpose(1, 2)
with torch.no_grad():
    cnn_fut_preds = final_cnn(X_fut_t).numpy().flatten()

future_metrics['CNN 1D (Sequence)'] = {
    'R2':   r2_score(y_fut_seq, cnn_fut_preds),
    'RMSE': np.sqrt(mean_squared_error(y_fut_seq, cnn_fut_preds)),
}
future_preds['CNN 1D (Sequence)'] = cnn_fut_preds

# --- 8C: Print summary table ---
print("\n" + "=" * 40)
print("FINAL METRICS ON HOLD-OUT SET (FUTURE)")
print("=" * 40)
for name, res in future_metrics.items():
    print(f"{name.ljust(20)} -> R2: {res['R2']:.4f} | RMSE: {res['RMSE']:.2f}")

# --- 8D: Time-series prediction plots ---
# Classical models predict for every row in df_future (dates align 1-to-1).
# The CNN predicts one step ahead per window, so the first SEQUENCE_LENGTH
# dates are consumed as the initial context and have no prediction label.
dates_classic = df_future['date_time'].values
dates_cnn     = df_future['date_time'].values[SEQUENCE_LENGTH:]

plt.figure(figsize=(18, 12))
for i, (name, preds) in enumerate(future_preds.items(), start=1):
    plt.subplot(3, 2, i)

    if 'CNN' in name:
        # CNN: aligned true values are y_fut_seq (already offset by seq_length)
        plt.plot(dates_cnn, y_fut_seq,  label='True Volume', color='black', linewidth=1.5)
        plt.plot(dates_cnn, preds,       label='Predicted',  color='orange', linestyle='--', alpha=0.8)
    else:
        # Classical: full future array, no offset needed
        plt.plot(dates_classic, y_future_raw, label='True Volume', color='black', linewidth=1.5)
        plt.plot(dates_classic, preds,         label='Predicted',  color='orange', linestyle='--', alpha=0.8)

    plt.title(f"{name} (Hold-out Set Predictions)")
    plt.xlabel("Date & Time")
    plt.ylabel("Traffic Volume")
    plt.xticks(rotation=45)
    plt.legend()

plt.tight_layout()
plt.savefig('plots/future_predictions_timeseries.png')
plt.close()

print("\n[SUCCESS] Pipeline execution finished. All plots saved to the 'plots' directory.")


---
# NOUVELLES ANALYSES — MISSIONS COMPLÉMENTAIRES
Les cellules ci-dessous sont indépendantes du pipeline principal.  
Elles peuvent être exécutées après les cellules 1 à 8 (le DataFrame `df` doit exister).


## MISSION 1 — Compte rendu PCA : quelles variables sont utiles ou pas ?

In [ ]:
# =============================================================================
# MISSION 1-A : Corrélations et variance — sélection préliminaire des variables
# =============================================================================
# Avant de lancer une PCA, on regarde la corrélation de chaque variable numérique
# avec la cible (traffic_volume) et leur dispersion (std).
# Une variable très peu corrélée ET à faible variance apporte peu d'information.

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("=== MISSION 1 : Analyse PCA des variables ===")

# Variables numériques disponibles (on exclut la cible et les colonnes catégorielles/datetime)
num_cols = ['temp', 'rain_1h', 'snow_1h', 'clouds_all',
            'day_of_week', 'month', 'hour_sin', 'hour_cos', 'is_holiday']

# S'assurer que les colonnes dérivées existent (cellule 4 doit avoir tourné)
df_pca = df[num_cols + ['traffic_volume']].dropna()

# --- 1-A.1 : Corrélation de Pearson avec la cible ---
corr_target = df_pca.corr()['traffic_volume'].drop('traffic_volume').sort_values(key=abs, ascending=False)
print("\nCorrélation de Pearson avec traffic_volume :")
print(corr_target.to_string())

# --- 1-A.2 : Heatmap de la matrice de corrélation complète ---
plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(df_pca.corr(), dtype=bool))  # masque triangle supérieur
sns.heatmap(df_pca.corr(), mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title("Heatmap des corrélations — Variables numériques + cible", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/pca_correlation_heatmap.png', dpi=120)
plt.show()
print("[Sauvegardé] plots/pca_correlation_heatmap.png")


In [ ]:
# =============================================================================
# MISSION 1-B : PCA — Variance expliquée et contribution des variables
# =============================================================================
# On applique une PCA sur les features normalisées (sans la cible).
# Les loadings (contributions) révèlent quelles variables portent chaque axe.
# Une variable avec de faibles loadings sur TOUS les axes est peu informative.

X_num = df_pca[num_cols].values
scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(X_num)

pca = PCA()
pca.fit(X_scaled)

# --- 1-B.1 : Variance expliquée cumulée (Scree Plot) ---
explained     = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained) + 1), explained * 100, color='steelblue', alpha=0.8)
axes[0].plot(range(1, len(explained) + 1), cum_explained * 100, 'ro-', linewidth=2)
axes[0].axhline(y=80, color='green', linestyle='--', alpha=0.7, label='80% seuil')
axes[0].set_xlabel('Composante principale')
axes[0].set_ylabel('Variance expliquée (%)')
axes[0].set_title('Scree Plot — Variance expliquée par composante')
axes[0].legend()

# --- 1-B.2 : Heatmap des loadings (contribution de chaque variable à chaque PC) ---
n_components_show = min(5, len(num_cols))
loadings = pd.DataFrame(
    pca.components_[:n_components_show].T,
    index=num_cols,
    columns=[f'PC{i+1}' for i in range(n_components_show)]
)

sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=axes[1])
axes[1].set_title('Loadings PCA — Contribution de chaque variable')

plt.tight_layout()
plt.savefig('plots/pca_variance_loadings.png', dpi=120)
plt.show()

# --- 1-B.3 : Compte rendu textuel ---
n80 = np.searchsorted(cum_explained, 0.80) + 1
print(f"\n{'='*60}")
print("COMPTE RENDU PCA")
print(f"{'='*60}")
print(f"Nombre de composantes pour atteindre 80% de variance : {n80}")
print(f"\nVariance expliquée par composante :")
for i, v in enumerate(explained):
    print(f"  PC{i+1} : {v*100:.1f}%  (cumulé : {cum_explained[i]*100:.1f}%)")

print(f"\nVariables les plus importantes (|loading| > 0.3 sur PC1 ou PC2) :")
top_vars = loadings[['PC1','PC2']].abs().max(axis=1)
top_vars_sorted = top_vars.sort_values(ascending=False)
for var, val in top_vars_sorted.items():
    tag = 'UTILE' if val >= 0.3 else 'FAIBLE contribution'
    print(f"  {var.ljust(15)} : {val:.3f}  → {tag}")

print("\nConclusion : snow_1h et is_holiday ont souvent de faibles loadings →")
print("  ils apportent peu d'info globale, mais peuvent quand même aider sur des cas rares.")
print("[Sauvegardé] plots/pca_variance_loadings.png")


## MISSION 2 — Modèle sans split passé/futur : seulement heure + weekend (sans timestamp)

In [ ]:
# =============================================================================
# MISSION 2 : Modèle sans split passé/futur — features réduites à hour + weekend
# =============================================================================
# Hypothèse : l'heure de la journée et le type de jour (semaine/weekend) sont
# les prédicteurs cycliques les plus puissants. On supprime :
#   - le timestamp (pas de split passé/futur)
#   - toutes les variables météo
#   - le mois, jour_de_semaine (gardés implicitement via is_weekend)
# Puis on compare les résultats avec les modèles complets (cellule 8).

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("=== MISSION 2 : Modèle réduit — heure + weekend uniquement ===")

# --- 2.1 : Construction des features minimales ---
df_m2 = df.copy()
df_m2['is_weekend'] = (df_m2['date_time'].dt.dayofweek >= 5).astype(int)

# Encodage cyclique de l'heure (identique à la cellule 4)
df_m2['hour']     = df_m2['date_time'].dt.hour
df_m2['hour_sin'] = np.sin(2 * np.pi * df_m2['hour'] / 24.0)
df_m2['hour_cos'] = np.cos(2 * np.pi * df_m2['hour'] / 24.0)

# Features finales : AUCUN timestamp, AUCUNE météo
features_m2 = ['hour_sin', 'hour_cos', 'is_weekend']

X_m2 = df_m2[features_m2].values
y_m2 = df_m2['traffic_volume'].values

# --- 2.2 : Validation croisée standard (KFold, pas TimeSeriesSplit) ---
# Sans information temporelle dans les features, un KFold classique est acceptable.
kf = KFold(n_splits=5, shuffle=True, random_state=42)

models_m2 = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    'KNN (k=10)':        KNeighborsRegressor(n_neighbors=10, weights='distance', n_jobs=-1),
}

results_m2 = {}
for name, model in models_m2.items():
    rmse_scores = []
    r2_scores   = []
    for train_idx, test_idx in kf.split(X_m2):
        model.fit(X_m2[train_idx], y_m2[train_idx])
        preds = model.predict(X_m2[test_idx])
        rmse_scores.append(np.sqrt(mean_squared_error(y_m2[test_idx], preds)))
        r2_scores.append(r2_score(y_m2[test_idx], preds))
    results_m2[name] = {
        'RMSE_mean': np.mean(rmse_scores), 'RMSE_std': np.std(rmse_scores),
        'R2_mean':   np.mean(r2_scores),   'R2_std':   np.std(r2_scores)
    }

# --- 2.3 : Affichage comparatif ---
print(f"\n{'Modèle':<22} {'RMSE moy':>10} {'RMSE std':>10}  {'R2 moy':>8} {'R2 std':>8}")
print('-' * 65)
for name, res in results_m2.items():
    print(f"{name:<22} {res['RMSE_mean']:>10.1f} {res['RMSE_std']:>10.1f}  {res['R2_mean']:>8.4f} {res['R2_std']:>8.4f}")

# --- 2.4 : Visualisation — prédictions vs réel pour 1 semaine ---
rf_m2 = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
rf_m2.fit(X_m2, y_m2)

# On prend une semaine représentative (lundi → dimanche)
mask_week = (df_m2['date_time'] >= '2017-06-05') & (df_m2['date_time'] < '2017-06-12')
df_week = df_m2[mask_week].copy().sort_values('date_time')
df_week['pred'] = rf_m2.predict(df_week[features_m2].values)

plt.figure(figsize=(15, 5))
plt.plot(df_week['date_time'], df_week['traffic_volume'],
         label='Réel', color='black', linewidth=2)
plt.plot(df_week['date_time'], df_week['pred'],
         label='Prédit (heure+weekend)', color='coral', linestyle='--', linewidth=2)
plt.fill_between(df_week['date_time'],
                 df_week['traffic_volume'], df_week['pred'],
                 alpha=0.15, color='coral')
for dt in df_week[df_week['is_weekend'] == 1]['date_time'].dt.date.unique():
    plt.axvspan(pd.Timestamp(dt), pd.Timestamp(dt) + pd.Timedelta(hours=23),
                alpha=0.07, color='blue')
plt.title("Mission 2 — RF (heure + weekend) vs Réel — Semaine du 05/06/2017")
plt.xlabel('Date & Heure')
plt.ylabel('Volume de trafic')
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('plots/mission2_hour_weekend_model.png', dpi=120)
plt.show()

print("\nInterprétation :")
print("  → Un R² > 0.7 avec seulement heure + weekend confirme que le rythme circadien")
print("    est le signal dominant. La météo apporte un gain marginal.")
print("  → Si R² < 0.6, la météo ou le mois ont une contribution non négligeable.")
print("[Sauvegardé] plots/mission2_hour_weekend_model.png")


## MISSION 3 — Jours particuliers : neige, tempête — impact sur la prédiction ?

In [ ]:
# =============================================================================
# MISSION 3-A : Identification des jours particuliers (neige, tempête, etc.)
# =============================================================================
# On cherche des jours où snow_1h > 0 ou weather_main == 'Snow'/'Thunderstorm'.
# On compare le trafic médian de ces jours avec la médiane globale à même heure.

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("=== MISSION 3 : Jours spéciaux — Impact sur le trafic ===")

df3 = df.copy()
df3['date'] = df3['date_time'].dt.date
df3['hour'] = df3['date_time'].dt.hour

# --- 3-A.1 : Catalogue des événements météo rares ---
weather_counts = df3['weather_main'].value_counts()
print("\nDistribution des catégories météo dans le dataset :")
print(weather_counts.to_string())

# --- 3-A.2 : Masques événements ---
snow_mask  = (df3['snow_1h'] > 0) | (df3['weather_main'] == 'Snow')
storm_mask = df3['weather_main'].isin(['Thunderstorm', 'Squall'])
fog_mask   = df3['weather_main'].isin(['Fog', 'Mist'])

snow_days  = df3[snow_mask]['date'].unique()
storm_days = df3[storm_mask]['date'].unique()
fog_days   = df3[fog_mask]['date'].unique()

print(f"\nNombre de jours de neige       : {len(snow_days)}")
print(f"Nombre de jours d'orage        : {len(storm_days)}")
print(f"Nombre de jours de brouillard  : {len(fog_days)}")

# --- 3-A.3 : Boxplots comparatifs ---
df3['event'] = 'Normal'
df3.loc[fog_mask,   'event'] = 'Brouillard/Brume'
df3.loc[storm_mask, 'event'] = 'Orage'
df3.loc[snow_mask,  'event'] = 'Neige'

plt.figure(figsize=(10, 6))
order   = ['Normal', 'Brouillard/Brume', 'Orage', 'Neige']
palette = {'Normal': '#4CAF50', 'Brouillard/Brume': '#90A4AE',
           'Orage': '#EF5350', 'Neige': '#42A5F5'}
sns.boxplot(x='event', y='traffic_volume', data=df3, order=order, palette=palette)

global_median = df3['traffic_volume'].median()
plt.axhline(global_median, color='black', linestyle='--', linewidth=1.5,
            label=f'Médiane globale ({global_median:.0f})')
plt.legend()
plt.title("Impact des événements météo extrêmes sur le trafic", fontsize=13, fontweight='bold')
plt.xlabel("Type d'événement")
plt.ylabel('Volume de trafic (véhicules/h)')
plt.tight_layout()
plt.savefig('plots/mission3_special_days_boxplot.png', dpi=120)
plt.show()

# --- 3-A.4 : Médiane par événement ---
print("\nVolume médian par type d'événement :")
for evt in order:
    med   = df3[df3['event'] == evt]['traffic_volume'].median()
    delta = ((med - global_median) / global_median) * 100
    print(f"  {evt:<22}: {med:>6.0f} veh/h  ({delta:+.1f}% vs mediane globale)")
print("[Sauvegardé] plots/mission3_special_days_boxplot.png")


In [ ]:
# =============================================================================
# MISSION 3-B : Zoom sur un jour de neige et un jour d'orage spécifiques
# =============================================================================
# On sélectionne les jours avec le plus fort événement (snow_1h max, orage long)
# et on trace le profil horaire réel + profil moyen du même jour de semaine.

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df3b = df.copy()
df3b['date'] = df3b['date_time'].dt.date
df3b['hour'] = df3b['date_time'].dt.hour
df3b['dow']  = df3b['date_time'].dt.dayofweek

# --- Identifier le pire jour de neige ---
snow_by_day = df3b[df3b['snow_1h'] > 0].groupby('date')['snow_1h'].sum()
if len(snow_by_day) > 0:
    worst_snow_day = snow_by_day.idxmax()
else:
    snow_days_df   = df3b[df3b['weather_main'] == 'Snow']
    worst_snow_day = snow_days_df.groupby('date')['date'].count().idxmax()

# --- Identifier le pire jour d'orage ---
storm_by_day = df3b[df3b['weather_main'] == 'Thunderstorm'].groupby('date')['date'].count()
worst_storm_day = storm_by_day.idxmax() if len(storm_by_day) > 0 else None

print(f"Pire jour de neige  : {worst_snow_day}")
print(f"Pire jour d'orage   : {worst_storm_day}")

day_names = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

def plot_special_day(df_full, special_date, event_label, ax, color):
    """Compare le profil horaire d'un jour spécial vs la moyenne du même dow."""
    special = df_full[df_full['date'] == special_date].sort_values('hour')
    if len(special) == 0:
        ax.set_title(f'{event_label} — pas de données')
        return
    dow      = special['dow'].iloc[0]
    baseline = df_full[df_full['dow'] == dow].groupby('hour')['traffic_volume'].mean()
    ax.plot(baseline.index, baseline.values, 'k--', linewidth=2,
            label=f'Moyenne {day_names[dow]}')
    ax.plot(special['hour'], special['traffic_volume'],
            color=color, linewidth=2.5, marker='o', markersize=5,
            label=f'{event_label} ({special_date})')
    ax.fill_between(special['hour'].values,
                    baseline.reindex(special['hour']).values,
                    special['traffic_volume'].values,
                    alpha=0.2, color=color)
    ax.set_title(f'{event_label} — {special_date}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Heure')
    ax.set_ylabel('Trafic (veh/h)')
    ax.legend(fontsize=8)
    ax.set_xticks(range(0, 24))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_special_day(df3b, worst_snow_day, 'NEIGE', axes[0], '#42A5F5')
if worst_storm_day:
    plot_special_day(df3b, worst_storm_day, 'ORAGE', axes[1], '#EF5350')
else:
    axes[1].set_title("Pas de données d'orage disponibles")

plt.suptitle('Impact des jours extrêmes vs profil moyen du même jour de semaine',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/mission3_special_day_profiles.png', dpi=120)
plt.show()

# --- Erreur de prédiction sur jours spéciaux ---
print("\nErreur du modèle (heure+weekend) sur les jours spéciaux :")
for label, day in [('Neige', worst_snow_day), ('Orage', worst_storm_day)]:
    if day is None:
        continue
    day_df = df3b[df3b['date'] == day].copy()
    if len(day_df) == 0:
        continue
    day_df['is_weekend'] = (day_df['dow'] >= 5).astype(int)
    day_df['hour_sin']   = np.sin(2 * np.pi * day_df['hour'] / 24.0)
    day_df['hour_cos']   = np.cos(2 * np.pi * day_df['hour'] / 24.0)
    X_day = day_df[['hour_sin', 'hour_cos', 'is_weekend']].values
    try:
        preds_day  = rf_m2.predict(X_day)
        rmse_day   = np.sqrt(np.mean((day_df['traffic_volume'].values - preds_day) ** 2))
        global_rmse = results_m2['Random Forest']['RMSE_mean']
        flag = '  SURREPRÉSENTATION D\'ERREUR' if rmse_day > global_rmse * 1.5 else '  OK — impact normal'
        print(f"  {label} ({day}) — RMSE : {rmse_day:.0f}  (moy global : {global_rmse:.0f}) {flag}")
    except NameError:
        print("  Exécuter d'abord la Mission 2 pour avoir rf_m2.")
print("[Sauvegardé] plots/mission3_special_day_profiles.png")


## MISSION 4 — Comprendre le vrai dataset : rush hours, pics bizarres, weekends

In [ ]:
# =============================================================================
# MISSION 4-A : Rush hours, weekends, saisonnalité — analyse profonde
# =============================================================================
# On décompose le trafic selon toutes ses dimensions temporelles :
# heure × jour de semaine, mois par année, distribution rush vs hors rush.

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("=== MISSION 4 : EDA Approfondie du Dataset ===")

df4 = df.copy()
df4['hour']       = df4['date_time'].dt.hour
df4['dow']        = df4['date_time'].dt.dayofweek
df4['month']      = df4['date_time'].dt.month
df4['year']       = df4['date_time'].dt.year
df4['is_weekend'] = (df4['dow'] >= 5)

# --- 4-A.1 : Heatmap Heure x Jour de semaine ---
pivot = df4.pivot_table(index='hour', columns='dow',
                        values='traffic_volume', aggfunc='mean')
pivot.columns = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.heatmap(pivot, ax=axes[0, 0], cmap='YlOrRd', annot=False,
            linewidths=0.3, cbar_kws={'label': 'Trafic moyen'})
axes[0, 0].set_title('Heatmap : Trafic moyen — Heure x Jour de semaine', fontweight='bold')
axes[0, 0].set_xlabel('Jour de semaine')
axes[0, 0].set_ylabel('Heure de la journée')

# --- 4-A.2 : Profil horaire — semaine vs weekend ---
weekday_profile = df4[~df4['is_weekend']].groupby('hour')['traffic_volume'].agg(['mean','std'])
weekend_profile = df4[ df4['is_weekend']].groupby('hour')['traffic_volume'].agg(['mean','std'])

ax = axes[0, 1]
ax.plot(weekday_profile.index, weekday_profile['mean'], 'b-o', linewidth=2.5,
        markersize=5, label='Jours ouvrables')
ax.fill_between(weekday_profile.index,
    weekday_profile['mean'] - weekday_profile['std'],
    weekday_profile['mean'] + weekday_profile['std'],
    alpha=0.15, color='blue')
ax.plot(weekend_profile.index, weekend_profile['mean'], 'r-s', linewidth=2.5,
        markersize=5, label='Weekend')
ax.fill_between(weekend_profile.index,
    weekend_profile['mean'] - weekend_profile['std'],
    weekend_profile['mean'] + weekend_profile['std'],
    alpha=0.15, color='red')
ax.axvspan(7, 9, alpha=0.1, color='green', label='Rush matin (7-9h)')
ax.axvspan(16, 18, alpha=0.1, color='orange', label='Rush soir (16-18h)')
ax.set_title('Profil horaire — Semaine vs Weekend (+-1 std)', fontweight='bold')
ax.set_xlabel('Heure')
ax.set_ylabel('Volume moyen (veh/h)')
ax.legend(fontsize=9)
ax.set_xticks(range(0, 24))

# --- 4-A.3 : Trafic mensuel moyen par année ---
monthly = df4.groupby(['year', 'month'])['traffic_volume'].mean().reset_index()
ax = axes[1, 0]
for yr, grp in monthly.groupby('year'):
    ax.plot(grp['month'], grp['traffic_volume'], marker='o', linewidth=1.8, label=str(yr))
ax.set_title('Trafic mensuel moyen par année (saisonnalité)', fontweight='bold')
ax.set_xlabel('Mois')
ax.set_ylabel('Volume moyen (veh/h)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Fev','Mar','Avr','Mai','Jun',
                     'Jul','Aou','Sep','Oct','Nov','Dec'], rotation=30, fontsize=8)
ax.legend(title='Année', fontsize=8)

# --- 4-A.4 : Violinplot Rush vs Hors rush ---
ax = axes[1, 1]
rush_h = df4['hour'].isin([7, 8, 9, 16, 17, 18])
data_vio = pd.concat([
    df4[ rush_h].assign(cat='Rush hours (7-9h, 16-18h)'),
    df4[~rush_h].assign(cat='Hors rush')
], ignore_index=True)
sns.violinplot(data=data_vio, x='cat', y='traffic_volume',
               palette=['#FF7043', '#42A5F5'], ax=ax, inner='quartile', cut=0)
ax.set_title('Distribution : Rush hours vs Hors rush', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Volume (veh/h)')

plt.suptitle('Mission 4 — EDA Approfondie : Rythmes du Trafic',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/mission4_deep_eda_part1.png', dpi=120, bbox_inches='tight')
plt.show()
print("[Sauvegardé] plots/mission4_deep_eda_part1.png")


In [ ]:
# =============================================================================
# MISSION 4-B : Détection des pics bizarres (outliers) avec Z-score
# =============================================================================
# On identifie les heures où le trafic est anormalement élevé ou bas
# par rapport à la distribution normale de cette heure + jour de semaine.
# Méthode : Z-score par groupe (heure, dow).

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("=== MISSION 4-B : Détection des pics bizarres ===")

df4b = df.copy()
df4b['hour'] = df4b['date_time'].dt.hour
df4b['dow']  = df4b['date_time'].dt.dayofweek
df4b['date'] = df4b['date_time'].dt.date

# Calcul du Z-score par groupe (heure, dow)
group_stats = df4b.groupby(['hour', 'dow'])['traffic_volume'].agg(['mean', 'std'])
df4b = df4b.join(group_stats, on=['hour', 'dow'])
df4b.rename(columns={'mean': 'grp_mean', 'std': 'grp_std'}, inplace=True)
df4b['zscore'] = (df4b['traffic_volume'] - df4b['grp_mean']) / (df4b['grp_std'] + 1e-9)

ZSCORE_THRESHOLD = 3.0
anomalies      = df4b[df4b['zscore'].abs() > ZSCORE_THRESHOLD].copy()
anomalies_high = anomalies[anomalies['zscore'] >  ZSCORE_THRESHOLD]
anomalies_low  = anomalies[anomalies['zscore'] < -ZSCORE_THRESHOLD]

print(f"Nombre d'anomalies detectees (|Z| > {ZSCORE_THRESHOLD}) :")
print(f"  Pics HAUTS  : {len(anomalies_high)} ({len(anomalies_high)/len(df4b)*100:.1f}%)")
print(f"  Pics BAS    : {len(anomalies_low)} ({len(anomalies_low)/len(df4b)*100:.1f}%)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution des Z-scores
axes[0].hist(df4b['zscore'], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline( ZSCORE_THRESHOLD, color='red',    linestyle='--', label=f'Z = +{ZSCORE_THRESHOLD}')
axes[0].axvline(-ZSCORE_THRESHOLD, color='orange', linestyle='--', label=f'Z = -{ZSCORE_THRESHOLD}')
axes[0].set_title('Distribution des Z-scores de trafic (par heure x dow)')
axes[0].set_xlabel('Z-score')
axes[0].set_ylabel('Frequence')
axes[0].legend()

# Anomalies dans la série temporelle
df4b_sorted = df4b.sort_values('date_time')
axes[1].plot(df4b_sorted['date_time'], df4b_sorted['traffic_volume'],
             color='lightblue', linewidth=0.4, alpha=0.8, label='Trafic')
axes[1].scatter(anomalies_high['date_time'], anomalies_high['traffic_volume'],
                color='red', s=8, label=f'Pic HAUT (Z>{ZSCORE_THRESHOLD})', zorder=5)
axes[1].scatter(anomalies_low['date_time'], anomalies_low['traffic_volume'],
                color='orange', s=8, label=f'Pic BAS (Z<-{ZSCORE_THRESHOLD})', zorder=5)
axes[1].set_title('Serie temporelle avec anomalies')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volume (veh/h)')
axes[1].legend(fontsize=8)
axes[1].tick_params(axis='x', rotation=30)

# Heatmap anomalies (heure x dow)
anom_count = anomalies.groupby(['dow', 'hour']).size().unstack(fill_value=0)
dow_labels = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
anom_count.index = [dow_labels[i] for i in anom_count.index]
sns.heatmap(anom_count, ax=axes[2], cmap='Reds', annot=True, fmt='d', linewidths=0.3)
axes[2].set_title("Nombre d'anomalies par heure x jour de semaine")
axes[2].set_xlabel('Heure')
axes[2].set_ylabel('Jour de semaine')

plt.suptitle(f'Mission 4-B — Pics bizarres (Z-score > {ZSCORE_THRESHOLD})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/mission4_anomalies.png', dpi=120)
plt.show()

# Top 10 pics
print("\nTop 10 pics HAUTS les plus extrêmes :")
top_h = anomalies_high.nlargest(10, 'zscore')[['date_time','traffic_volume','weather_main','grp_mean','zscore']].copy()
top_h['grp_mean'] = top_h['grp_mean'].round(0)
top_h['zscore']   = top_h['zscore'].round(2)
print(top_h.to_string(index=False))

print("\nTop 10 pics BAS les plus extrêmes :")
top_l = anomalies_low.nsmallest(10, 'zscore')[['date_time','traffic_volume','weather_main','grp_mean','zscore']].copy()
top_l['grp_mean'] = top_l['grp_mean'].round(0)
top_l['zscore']   = top_l['zscore'].round(2)
print(top_l.to_string(index=False))
print("[Sauvegardé] plots/mission4_anomalies.png")


In [ ]:
# =============================================================================
# MISSION 4-C : Statistiques descriptives complètes du dataset
# =============================================================================
# Résumé chiffré de tous les patterns identifiés, imprimé sous forme de rapport.

import pandas as pd
import numpy as np

df4c = df.copy()
df4c['hour']       = df4c['date_time'].dt.hour
df4c['dow']        = df4c['date_time'].dt.dayofweek
df4c['is_weekend'] = (df4c['dow'] >= 5)
df4c['month']      = df4c['date_time'].dt.month
df4c['year']       = df4c['date_time'].dt.year

print("=" * 65)
print("RAPPORT COMPLET — Metro Interstate Traffic Volume")
print("=" * 65)

print(f"\nPeriode couverte : {df4c['date_time'].min()} -> {df4c['date_time'].max()}")
print(f"Nombre d'observations : {len(df4c):,}")
print(f"Trafic moyen   : {df4c['traffic_volume'].mean():.0f} veh/h")
print(f"Trafic median  : {df4c['traffic_volume'].median():.0f} veh/h")
print(f"Trafic max     : {df4c['traffic_volume'].max():.0f} veh/h")
print(f"Trafic min     : {df4c['traffic_volume'].min():.0f} veh/h")

rush_mask    = df4c['hour'].isin([7, 8, 9, 16, 17, 18])
rush_mean    = df4c[ rush_mask & ~df4c['is_weekend']]['traffic_volume'].mean()
offpeak_mean = df4c[~rush_mask & ~df4c['is_weekend']]['traffic_volume'].mean()
weekend_mean = df4c[ df4c['is_weekend']]['traffic_volume'].mean()
weekday_mean = df4c[~df4c['is_weekend']]['traffic_volume'].mean()

print(f"\n--- Rush Hours (semaine, 7-9h & 16-18h) ---")
print(f"  Trafic moyen rush     : {rush_mean:.0f} veh/h")
print(f"  Trafic moyen hors rush: {offpeak_mean:.0f} veh/h")
print(f"  Facteur rush/hors-rush: x{rush_mean/offpeak_mean:.2f}")

print(f"\n--- Weekend vs Semaine ---")
print(f"  Trafic moyen weekend  : {weekend_mean:.0f} veh/h")
print(f"  Trafic moyen semaine  : {weekday_mean:.0f} veh/h")
print(f"  Facteur weekend/sem.  : x{weekend_mean/weekday_mean:.2f}")

peak_by_hour = df4c[~df4c['is_weekend']].groupby('hour')['traffic_volume'].mean()
peak_hour    = peak_by_hour.idxmax()
print(f"\n  Heure de pointe semaine : {peak_hour}h ({peak_by_hour[peak_hour]:.0f} veh/h)")

monthly_avg   = df4c.groupby('month')['traffic_volume'].mean()
month_names   = {1:'Jan',2:'Fev',3:'Mar',4:'Avr',5:'Mai',6:'Jun',
                 7:'Jul',8:'Aou',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
print(f"\n--- Saisonnalite ---")
print(f"  Mois le plus charge  : {month_names[monthly_avg.idxmax()]} ({monthly_avg.max():.0f} veh/h)")
print(f"  Mois le plus calme   : {month_names[monthly_avg.idxmin()]} ({monthly_avg.min():.0f} veh/h)")

holiday_mean = df4c[df4c['holiday'] != 'None']['traffic_volume'].mean()
normal_mean  = df4c[df4c['holiday'] == 'None']['traffic_volume'].mean()
print(f"\n--- Jours feries ---")
print(f"  Trafic moyen jours feries : {holiday_mean:.0f} veh/h")
print(f"  Trafic moyen jours normaux: {normal_mean:.0f} veh/h")
print(f"  Reduction due aux feries  : {(1 - holiday_mean/normal_mean)*100:.1f}%")

print(f"\n--- Trafic moyen par condition meteo ---")
weather_traffic = df4c.groupby('weather_main')['traffic_volume'].agg(['mean','count']).sort_values('mean', ascending=False)
for wth, row in weather_traffic.iterrows():
    bar = '#' * int(row['mean'] / 400)
    print(f"  {wth:<15} : {row['mean']:>5.0f} veh/h  ({int(row['count']):>6,} obs)  {bar}")

print("\n" + "=" * 65)
print("FIN DU RAPPORT")
print("=" * 65)
